# Truncated Single Value Decomposition

We will work through how to use TSVD to get higher dimensional data to lower dimensions.

These methods we will go through are not necessarily the fastest or most efficient ways but they should give you a good feeling for how it works. And there is nothing stopping you optimising the methods we will explore, or writing something even faster. They also use some of the most basic Python libraries, so are simple to impliment by design.

Let's include our libraries.

In [ ]:
import numpy as np              #To make some operations easier
import matplotlib.pyplot as plt #To see what we are doing

## Single Value Decomposition

For TSVD, we need to start with SVD:

Any matrix __M__ can be decomposed into a unitary matrix (__U__), a diagonal matrix (__Σ__), and the transpose of a unitary matrix (__V__<sup>T</sup>). So we can rewrite matricies as combinations of other matricies.

__Σ__ contains the ordered eigenvalues of __M__

__V__ contains the ordered eigenvectors of __M__

We can then use __Σ__ and __Σ__ to reduce the dimensionality of our data to _d_ dimensions by using the first _d_ eigenvectors and eigenvalues.

Let's get a toy matrix to play with

In [ ]:
M = np.array([[8,1,3],
              [5,1,9]])

Lets start by finding __U__ (which is the transpose of __V__). We will also get __Σ__ at the same time.

Remember, __U__ should be eigenvectors of __MM__<sup>T</sup> (which you will also want to keep for later) and __Σ__ is made from the _sorted_ eigenvalues.

In [ ]:
def get_U(M):
    
    B_U = np.dot(M, M.T)
    #If you want, numpy can get your eigenvalues and eigenvectors
    eigenvalues, eigenvectors = np.linalg.eig(B_U)

    #You can fill out the rest
    
    return U, Sigma_U, B_U

U, Sigma_U, B_U = get_U(M)

We can also get __V__<sup>T</sup> in a similar way. This is the eigenvectors of __M__<sup>T</sup>__M__ (which you will also want to keep). Also calculate Sigma again.

In [ ]:
def get_Vt(M):
    
    return Vt, Sigma_Vt, B_Vt

Vt, Sigma_Vt, B_Vt = get_Vt(M)

We now have __U__, __V__<sup>T</sup>, and __Σ__ twice. We only want one __Σ__, specifally the __Σ__ we made from the largest __B__ (which is why we kept them.

Let's wrap this all together nicely.

In [ ]:
def get_U_Sigma_Vt(M):
    U, Sigma_U, B_U = get_U(M)
    Vt, Sigma_Vt, B_Vt = get_Vt(M)

    #You can fill out the rest
    
    return U, Sigma, Vt

U, Sigma, Vt = get_U_Sigma_Vt(M)

Let's check it worked. We can reconstruct __M__ with __U__, __Σ__, and __V__<sup>T</sup>

In [ ]:
M_reconstructed = np.dot(np.dot(U, Sigma), Vt)

print(M)
print()
print(M_reconstructed)

The __Σ__ is the wrong shape, huh... We can clip it down or add rows or columns with 0s to fix that.

One way to do it is to make an array of 0s the shape of __M__, and fill the top left with the __Σ__ you made. Update your functions and try again.

In [ ]:
M_reconstructed = np.dot(np.dot(U, Sigma), Vt)

print(M)
print()
print(M_reconstructed)

Look how __M__ and __M__<sub>reconstructed</sub> are the same... Or... Well...

If you used np.linalg.eig, the eigen values you get can have different signs when you run it in get_U() and get_Vt(). The only way to avoid this is to get __U__ and __V__<sup>T</sup> at the same time. We can calculate __U__ (or __V__<sup>T</sup>) and then calculate __V__<sup>T</sup> (or __U__) from that.

In [ ]:
def get_U_Sigma_Vt_fixed(M):

    return U, Sigma, Vt

U, Sigma, Vt = get_U_Sigma_Vt_fixed(M)

Now lets check we can reconstruct __M__ properly this time.

In [ ]:
M_reconstructed = np.dot(np.dot(U, Sigma), Vt)

print(M)
print()
print(M_reconstructed)

YAY!

## Truncated SVD

We have our single values. The higer the eigenvalue, the more information is carried in that eigen vector. This allows us to discard the eigenvectors associated with the smaller eigenvalues without losing anything very important.

We now need some data to play with, so let's make some 6D data that we want to drop into 2D space.

In [ ]:
#While we play around, we want our results to be repeatable. So let's set a seed
np.random.seed(256)

#Draw 20 random points from a 3D Gaussian with
point_1 = np.random.multivariate_normal([2,2,6], [[0.75,0.00,0.00],[0.00,0.50,0.00],[0.00,0.00,0.25]], 20) #mean=[2,2,6] and std=[0.75,0.50,0.25]
point_2 = np.random.multivariate_normal([6,6,4], [[0.50,0.00,0.00],[0.00,0.75,0.00],[0.00,0.00,0.25]], 20) #mean=[6,6,4] and std=[0.50,0.75,0.25]
point_3 = np.random.multivariate_normal([4,4,2], [[0.25,0.00,0.00],[0.00,0.25,0.00],[0.00,0.00,0.75]], 20) #mean=[4,4,2] and std=[0.25,0.25,0.75]
points = np.vstack((point_1, point_2, point_3))
#Draw 3 values between 0 and 1 for each point (this are effectivly RGB colours, to help visualise 6D)
colour = np.random.uniform(low=[0,0,0], high=[1,1,1], size=[len(points),3]) #
X = np.hstack((points, colour))

fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.scatter(X[:,0], X[:,1], X[:,2], color=X[:,3:])
plt.show()

Remember, we can move to the lower _d_ dimensional space using __U__ and __Σ__ from SVD of __M__: __Z__=__U__<sub>_d_</sub>__Σ__<sub>_d_</sub>. We will also calculate and keep __V__<sup>T</sup><sub>_d_</sub> for later

In [ ]:
def tsvd(M, d=2):
    
    return Z, Vt_d

Z, Vt_d = tsvd(X)

plt.scatter(Z[:,0], Z[:,1])
plt.show()

To prove this is actually working we can check against scikit learn's TSVD

In [ ]:
from sklearn.decomposition import TruncatedSVD

skl_tsvd = TruncatedSVD()
skl_tsvd.fit(X)
Z_skl = skl_tsvd.transform(X)

plt.scatter(Z[:,0], Z[:,1], label="Yours")
plt.scatter(Z_skl[:,0], Z_skl[:,1], label='sklearn')
plt.legend(loc=0, frameon=False)
plt.show()

There is a good chance your points do not overlap with the ones from sklearn. If this is the case andyou check closely, they are mirred around x=0 and/or y=0. Those pesky signs on eignevalues are doing this, so it's not anything to worry about. You can flip the signs in your __Z__ space if you want to check.

Lets gets some more toy data to drop into our new reduced dimensional space.

In [ ]:
#Draw 20 random points from a 3D Gaussian with
point_1 = np.random.multivariate_normal([2,2,6], [[0.75,0.00,0.00],[0.00,0.50,0.00],[0.00,0.00,0.25]], 20) #mean=[2,2,6] and std=[0.75,0.50,0.25]
point_2 = np.random.multivariate_normal([6,6,4], [[0.50,0.00,0.00],[0.00,0.75,0.00],[0.00,0.00,0.25]], 20) #mean=[6,6,4] and std=[0.50,0.75,0.25]
point_3 = np.random.multivariate_normal([4,4,2], [[0.25,0.00,0.00],[0.00,0.25,0.00],[0.00,0.00,0.75]], 20) #mean=[4,4,2] and std=[0.25,0.25,0.75]
points = np.vstack((point_1, point_2, point_3))
#Draw 3 values between 0 and 1 for each point (this are effectivly RGB colours, to help visualise 6D)
colour = np.random.uniform(low=[0,0,0], high=[1,1,1], size=[len(points),3]) #
Y = np.hstack((points, colour))

fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.scatter(Y[:,0], Y[:,1], Y[:,2], color=Y[:,3:])
plt.show()

To drop this 6D data into out 2D space, we need to use the __V__<sup>T</sup><sub>_d_</sub> we saved earlier.

In [ ]:
Z_Y = np.dot(Y, Vt_d.T)

plt.scatter(Z[:,0], Z[:,1], label='Original')
plt.scatter(Z_Y[:,0], Z_Y[:,1], label='New')
plt.legend(loc=0, frameon=False)
plt.show()

Let's save our Z data for this afternoon.

In [ ]:
np.save('TSVD_Z.npy', Z)

## Real Data

Now let's play with some real data! MorphologicalCat_VISTULA.csv contains morphologies for 10943 galaxies observed by KiDS. Let's grab that data!

In [ ]:
#The first row is the header, so we will skip that
data = np.genfromtxt('MorphologicalCat_VISTULA.csv', dtype=float, delimiter=',')[1:]

Now run your TSVD on the data.

In [ ]:
Z_data, Vt_d = tsvd(data)

plt.hexbin(Z_data[:,0], Z_data[:,1], mincnt=1)
plt.show()

That looks like two clusters. Remember, the TSVD axes don't really tell us anything in and of themselves. So we don't know what these clusters are from the x and y axes. We would need to find the members of the clusters and eximin them closer. Let's save this 2D reduction for now.

In [ ]:
np.save('TSVD_data.npy', Z_data)